# 03 · Train the cloak generator

**Every code cell is self-contained.** On a fresh or reconnected runtime you
can run any one of them directly; you never need to rerun the cells above it.

- **Smoke run** (~20–30 min): 300 steps. Checks the loop works, and measures
  the real speed so it can project how long the main run will take.
- **Progress** (no GPU needed): status, ETA, loss and validation curves and the
  latest samples, read from Drive. Run it whenever, even while training runs
  in another session.
- **Main run**: checkpoints to Drive every 500 steps and stops itself before
  the Colab session limit (`budget_min` in the config). **After a disconnect or
  when it stops on the time budget, rerun only this cell**: it resumes from
  `last.pt` and loses at most the steps since the last checkpoint.

What it saves in `MyDrive/soteria/runs/<RUN_NAME>/`: `status.json` (step,
GPU-minutes, ETA, latest validation), `log.csv` (training curves), `val.csv`
(protection metrics every 500 steps), `samples/`, `generator_ema.pt` (the model
to use) and `last.pt` (full resume state).

Each person and each experiment needs its own `RUN_NAME`. Two runs writing to
one folder would resume from each other's checkpoints.

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python -m soteria.train --config configs/smoke.yaml out_dir={DRIVE_ROOT}/runs/smoke
import json, glob, pandas as pd, yaml
from IPython.display import Image as Show, display
log = pd.read_csv(f"{DRIVE_ROOT}/runs/smoke/log.csv")
img_per_s = log["img_per_s"].iloc[len(log) // 2:].median()     # skip warm-up
main = yaml.safe_load(open("configs/base_256.yaml"))
hours = main["steps"] * main["batch_size"] / img_per_s / 3600
sessions = hours * 60 / main["budget_min"]
print(f"{img_per_s:.1f} img/s -> main run ({main['steps']} steps x {main['batch_size']}) ~ {hours:.1f} GPU-hours "
      f"~ {sessions:.1f} sessions of {main['budget_min']} min")
display(Show(sorted(glob.glob(f"{DRIVE_ROOT}/runs/smoke/samples/*.png"))[-1]))

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=False, gpu=False, requirements=())
# -----------------------------------------------------------------------------

# ---- this run ----------------------------------------------------------------
RUN_NAME  = "base_256"     # one folder per experiment AND per person, e.g. "ahnaf_base_256"
CONFIG    = "configs/base_256.yaml"
OVERRIDES = ""             # e.g. "clip_floor=0.3 face_floor=-0.1 w_vae=4". Must not change
                           # while resuming one RUN_NAME: train.py refuses a mismatch.
OUT = f"{DRIVE_ROOT}/runs/{RUN_NAME}"

import glob, json, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image as Show, display
status = json.load(open(f"{OUT}/status.json"))
print(json.dumps({k: v for k, v in status.items() if k != "last_val"}, indent=1))
log = pd.read_csv(f"{OUT}/log.csv")
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
log.plot(x="step", y=[c for c in log if c.startswith("loss_")], ax=axes[0], title="train loss terms")
log.plot(x="step", y=[c for c in ("lpips_raw", "clip_cos", "face_cos") if c in log], ax=axes[1], title="train")
try:
    val = pd.read_csv(f"{OUT}/val.csv")
    val.plot(x="step", y=[c for c in val if c.startswith("jpeg75/") and not c.endswith("_shift")],
             ax=axes[2], title="val after JPEG q75 (lower = better)")
    display(val.tail(3).set_index("step").T.round(4))
except FileNotFoundError:
    axes[2].set_title("no validation yet")
plt.tight_layout(); plt.show()
samples = sorted(glob.glob(f"{OUT}/samples/*.png"))
if samples:
    print(samples[-1], "(rows: clean / cloaked / |delta| x10)")
    display(Show(samples[-1]))

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

# ---- this run ----------------------------------------------------------------
RUN_NAME  = "base_256"     # one folder per experiment AND per person, e.g. "ahnaf_base_256"
CONFIG    = "configs/base_256.yaml"
OVERRIDES = ""             # e.g. "clip_floor=0.3 face_floor=-0.1 w_vae=4". Must not change
                           # while resuming one RUN_NAME: train.py refuses a mismatch.
OUT = f"{DRIVE_ROOT}/runs/{RUN_NAME}"

!python -m soteria.train --config {CONFIG} out_dir={OUT} {OVERRIDES}
print(open(f"{OUT}/status.json").read())